# Fusou Datasets 可視化サンプル集
Pythonスクリプト版のサンプルをJupyterノートブックにまとめたものです。
キャッシュが無ければ自動ダウンロードして、各種グラフを描画します。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import fusou_datasets

def load_table(name: str, master: bool = False):
    try:
        return fusou_datasets.load_master(name) if master else fusou_datasets.load(name, offline=True)
    except Exception:
        return fusou_datasets.load_master(name) if master else fusou_datasets.load(name, offline=False)

fusou_datasets.configure(cache_dir="~/.fusou_datasets/cache")
print("✓ setup ready")

## 1. Battle件数のタイムライン

In [ ]:
df_battle = load_table("battle")
if "timestamp" in df_battle.columns:
    ts = pd.to_datetime(df_battle["timestamp"], unit="s")
    daily = ts.dt.floor("D").value_counts().sort_index()
    daily.plot(kind="bar", figsize=(10,4), color="#4C7EF3", title="Battle Count by Day")
    plt.xlabel("Date"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
else:
    print("timestamp column not found in battle table")

## 2. 艦種別レベル分布

In [ ]:
df_ship = load_table("own_ship")
df_mst_ship = load_table("mst_ship", master=True)
if {"ship_id", "lv"}.issubset(df_ship.columns):
    merged = df_ship.merge(df_mst_ship[["id", "stype"]], left_on="ship_id", right_on="id", how="left")
    plt.figure(figsize=(10,5))
    for stype, group in merged.groupby("stype"):
        group["lv"].plot(kind="kde", label=f"stype {stype}")
    plt.title("Ship Level Density by Ship Type")
    plt.xlabel("Level"); plt.ylabel("Density"); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("own_ship must have ship_id and lv columns")

## 3. 装備使用頻度トップN

In [ ]:
df_slot = load_table("own_slotitem")
df_mst_slot = load_table("mst_slotitem", master=True)
if "mst_slotitem_id" in df_slot.columns:
    top_n = 15
    counts = df_slot["mst_slotitem_id"].value_counts().head(top_n)
    merged = counts.rename_axis("mst_slotitem_id").reset_index(name="count")
    merged = merged.merge(df_mst_slot[["id", "name"]], left_on="mst_slotitem_id", right_on="id", how="left")
    merged.plot(kind="barh", x="name", y="count", figsize=(10,6), color="#2FB99B")
    plt.gca().invert_yaxis(); plt.title(f"Top {top_n} Slotitem Usage"); plt.xlabel("Count"); plt.tight_layout(); plt.show()
else:
    print("own_slotitem must have mst_slotitem_id column")

## 4. Hougeki 命中×ダメージ ヒートマップ

In [ ]:
try:
    df_houg = load_table("hougeki")
except Exception:
    df_houg = pd.DataFrame()
    print("hougeki table not available")
if not df_houg.empty and {"hit", "damage"}.issubset(df_houg.columns):
    bins = [0, 10, 30, 60, 100, 200, 400, 800]
    df_houg["damage_bin"] = pd.cut(df_houg["damage"], bins=bins, labels=["0-10","10-30","30-60","60-100","100-200","200-400","400+"])
    pivot = df_houg.pivot_table(index="hit", columns="damage_bin", values="damage", aggfunc="count", fill_value=0)
    plt.figure(figsize=(10,6))
    plt.imshow(pivot, aspect="auto", cmap="YlGnBu")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label="count")
    plt.title("Hougeki Hit vs Damage Bucket")
    plt.tight_layout(); plt.show()
else:
    print("hougeki needs hit and damage columns")

## 5. EnvInfo のタイムゾーン分布

In [ ]:
df_env = load_table("env_info")
if "timezone_offset" in df_env.columns:
    counts = df_env["timezone_offset"].value_counts().sort_index()
    counts.plot(kind="bar", figsize=(8,4), color="#F29E4C", title="EnvInfo Timezone Offset Distribution")
    plt.xlabel("Timezone offset (minutes)"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
else:
    print("env_info must have timezone_offset column")

## 6. セル遷移（同一マップ内フロー）

In [ ]:
df_cells = load_table("cells")
needed = {"battle_uuid", "mapinfo_no", "cell"}
if needed.issubset(df_cells.columns):
    df_cells = df_cells.sort_values(["battle_uuid", "timestamp"] if "timestamp" in df_cells.columns else ["battle_uuid"])
    df_cells["next_cell"] = df_cells.groupby("battle_uuid")["cell"].shift(-1)
    df_cells["next_map"] = df_cells.groupby("battle_uuid")["mapinfo_no"].shift(-1)
    flows = df_cells[(df_cells["next_cell"].notna()) & (df_cells["mapinfo_no"] == df_cells["next_map"])]
    flows["edge"] = flows["cell"].astype(str) + " → " + flows["next_cell"].astype(str)
    top_edges = flows["edge"].value_counts().head(20)
    top_edges.plot(kind="barh", figsize=(10,5), color="#6A7FDB", title="Top Cell-to-Cell Flows (same map)")
    plt.gca().invert_yaxis(); plt.xlabel("Count"); plt.tight_layout(); plt.show()
else:
    print("cells needs battle_uuid, mapinfo_no, cell columns")

## 7. 敵/友軍デッキの艦数バランス

In [ ]:
try:
    df_enemy = load_table("enemy_deck")
    df_friend = load_table("friend_deck")
except Exception:
    df_enemy = pd.DataFrame(); df_friend = pd.DataFrame(); print("enemy_deck or friend_deck not available")
def count_ships(df):
    if "ship_ids" not in df.columns:
        return pd.Series(dtype=int)
    return df["ship_ids"].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
enemy_counts = count_ships(df_enemy)
friend_counts = count_ships(df_friend)
plt.figure(figsize=(8,4))
bins = range(0, 13)
plt.hist(enemy_counts, bins=bins, alpha=0.6, label="Enemy", color="#E46464")
plt.hist(friend_counts, bins=bins, alpha=0.6, label="Friend", color="#4DB6AC")
plt.title("Fleet Size Distribution (Enemy vs Friend)")
plt.xlabel("Ship count per deck"); plt.ylabel("Frequency"); plt.legend(); plt.tight_layout(); plt.show()